This is the start of a new project

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import butter, filtfilt, find_peaks

In [20]:
DATA_DIR = Path(r"src")
CSV_FILE = DATA_DIR / "Running trial 2.csv"

CSV_FILE = DATA_DIR / "Running trial 2.csv"
mb = 59.8
g = 9.81
BW = mb * g
m1 = 0.08 * mb
m2 = 0.92 * mb

In [21]:
print("CSV file:", CSV_FILE)
print("Body mass =", mb)
print("Body weight =", BW)
print("m1 =", m1)
print("m2 =", m2)

CSV file: src\Running trial 2.csv
Body mass = 59.8
Body weight = 586.638
m1 = 4.784
m2 = 55.016


In [23]:
df = pd.read_csv(CSV_FILE)

print("Columns:")
for c in df.columns:
    print(c)

time = df["Time"].to_numpy()
fs = round(1 / np.mean(np.diff(time)))
dt = 1 / fs

print("\nSampling frequency =", fs)


Columns:
Frame
Time
mal_lat_left_X
mal_lat_left_Y
mal_lat_left_Z
calc_back_left_X
calc_back_left_Y
calc_back_left_Z
toe_left_X
toe_left_Y
toe_left_Z
mal_lat_right_X
mal_lat_right_Y
mal_lat_right_Z
calc_back_right_X
calc_back_right_Y
calc_back_right_Z
toe_right_X
toe_right_Y
toe_right_Z
C7_X
C7_Y
C7_Z

Sampling frequency = 200


In [24]:
heelL = df["calc_back_left_Z"].to_numpy()
toeL  = df["toe_left_Z"].to_numpy()
ankL  = df["mal_lat_left_Z"].to_numpy() / 1000

heelR = df["calc_back_right_Z"].to_numpy()
toeR  = df["toe_right_Z"].to_numpy()
ankR  = df["mal_lat_right_Z"].to_numpy() / 1000

fc = 25
b, a = butter(4, fc/(fs/2), btype="low")

heelZ_L = filtfilt(b, a, heelL)
toeZ_L  = filtfilt(b, a, toeL)
ankZ_L  = filtfilt(b, a, ankL)

heelZ_R = filtfilt(b, a, heelR)
toeZ_R  = filtfilt(b, a, toeR)
ankZ_R  = filtfilt(b, a, ankR)

print("Filtering done")

Filtering done


In [26]:
locs_L, _ = find_peaks(-heelZ_L, distance=round(0.30 * fs))
locs_R, _ = find_peaks(-heelZ_R, distance=round(0.30 * fs))

print("Left touchdown candidates =", len(locs_L))
print("Right touchdown candidates =", len(locs_R))
print("Total touchdown candidates =", len(locs_L) + len(locs_R))

Left touchdown candidates = 44
Right touchdown candidates = 45
Total touchdown candidates = 89


In [27]:
toeV_L = np.gradient(toeZ_L, dt)
toeV_R = np.gradient(toeZ_R, dt)

print("Toe velocity computed")


Toe velocity computed


In [29]:
def pair_full_steps(locs, toeV, time, fs):
    TD = []
    TO = []
    nextTD = []
    tc = []
    ta = []

    for i in range(len(locs) - 1):
        td = locs[i]
        td_next = locs[i + 1]

        search_start = td + round(0.12 * fs)
        search_end = min(td + round(0.35 * fs), td_next - 1)

        if search_end <= search_start:
            continue

        seg = toeV[search_start:search_end + 1]
        idx_max = np.argmax(seg)
        to = search_start + idx_max

        TD.append(time[td])
        TO.append(time[to])
        nextTD.append(time[td_next])
        tc.append((to - td) / fs)
        ta.append((td_next - to) / fs)

    return (
        np.array(TD),
        np.array(TO),
        np.array(nextTD),
        np.array(tc),
        np.array(ta),
    )

TD_L_full, TO_L_full, nextTD_L_full, tc_L_full, ta_L_full = pair_full_steps(locs_L, toeV_L, time, fs)
TD_R_full, TO_R_full, nextTD_R_full, tc_R_full, ta_R_full = pair_full_steps(locs_R, toeV_R, time, fs)

print("Left paired steps =", len(tc_L_full))
print("Right paired steps =", len(tc_R_full))
print("Total paired steps =", len(tc_L_full) + len(tc_R_full))

Left paired steps = 43
Right paired steps = 44
Total paired steps = 87


In [30]:
T_contacts_left_full = pd.DataFrame({
    "Step": np.arange(1, len(tc_L_full) + 1),
    "Touchdown_s": TD_L_full,
    "ToeOff_s": TO_L_full,
    "NextTouchdown_s": nextTD_L_full,
    "ContactTime_s": tc_L_full,
    "AerialTime_s": ta_L_full,
})

T_contacts_right_full = pd.DataFrame({
    "Step": np.arange(1, len(tc_R_full) + 1),
    "Touchdown_s": TD_R_full,
    "ToeOff_s": TO_R_full,
    "NextTouchdown_s": nextTD_R_full,
    "ContactTime_s": tc_R_full,
    "AerialTime_s": ta_R_full,
})

T_contacts_left_full.head()


,Step,Touchdown_s,ToeOff_s,NextTouchdown_s,ContactTime_s,AerialTime_s
0,1,0.405,0.685,1.075,0.280,0.390
1,2,1.075,1.370,1.745,0.295,0.375
2,3,1.745,2.030,2.425,0.285,0.395
3,4,2.425,2.715,3.095,0.290,0.380
4,5,3.095,3.400,3.775,0.305,0.375


In [31]:
def compute_dt1(contact_df, time, ankZ, fs):
    dt1 = []

    for _, row in contact_df.iterrows():
        td_time = row["Touchdown_s"]
        to_time = row["ToeOff_s"]

        td_idx = np.argmin(np.abs(time - td_time))
        to_idx = np.argmin(np.abs(time - to_time))

        start_idx = td_idx + max(1, round(0.010 * fs))   # 10 ms after touchdown
        if to_idx <= start_idx:
            dt1.append(np.nan)
            continue

        local = ankZ[start_idx:to_idx + 1]
        idx_local_min = np.argmin(local)
        min_idx = start_idx + idx_local_min

        dt1_val = time[min_idx] - time[td_idx]
        dt1.append(dt1_val)

    return np.array(dt1)

T_contacts_left_full["dt1_s"] = compute_dt1(T_contacts_left_full, time, ankZ_L, fs)
T_contacts_right_full["dt1_s"] = compute_dt1(T_contacts_right_full, time, ankZ_R, fs)

T_contacts_left_full.head()

,Step,Touchdown_s,ToeOff_s,NextTouchdown_s,ContactTime_s,AerialTime_s,dt1_s
0,1,0.405,0.685,1.075,0.280,0.390,0.01
1,2,1.075,1.370,1.745,0.295,0.375,0.01
2,3,1.745,2.030,2.425,0.285,0.395,0.01
3,4,2.425,2.715,3.095,0.290,0.380,0.01
4,5,3.095,3.400,3.775,0.305,0.375,0.01


In [32]:
ankV_L = np.gradient(ankZ_L, dt)
ankV_R = np.gradient(ankZ_R, dt)

def compute_dv1(contact_df, time, ankV, fs):
    dv1 = []

    for _, row in contact_df.iterrows():
        td_time = row["Touchdown_s"]
        td_idx = np.argmin(np.abs(time - td_time))

        start_pre = max(0, td_idx - round(0.020 * fs))   # 20 ms before touchdown
        v_down = np.min(ankV[start_pre:td_idx + 1])

        dv1.append(abs(v_down))

    return np.array(dv1)

T_contacts_left_full["dv1_mps"] = compute_dv1(T_contacts_left_full, time, ankV_L, fs)
T_contacts_right_full["dv1_mps"] = compute_dv1(T_contacts_right_full, time, ankV_R, fs)

T_contacts_left_full.head()

,Step,Touchdown_s,ToeOff_s,NextTouchdown_s,ContactTime_s,AerialTime_s,dt1_s,dv1_mps
0,1,0.405,0.685,1.075,0.280,0.390,0.01,0.330098
1,2,1.075,1.370,1.745,0.295,0.375,0.01,0.435685
2,3,1.745,2.030,2.425,0.285,0.395,0.01,0.284125
3,4,2.425,2.715,3.095,0.290,0.380,0.01,0.228049
4,5,3.095,3.400,3.775,0.305,0.375,0.01,0.331798


In [34]:
# article: t_step = tc + ta
T_contacts_left_full["t_step_s"] = T_contacts_left_full["ContactTime_s"] + T_contacts_left_full["AerialTime_s"]
T_contacts_right_full["t_step_s"] = T_contacts_right_full["ContactTime_s"] + T_contacts_right_full["AerialTime_s"]

# JT = mb * g * t_step
T_contacts_left_full["JT_Ns"] = mb * g * T_contacts_left_full["t_step_s"]
T_contacts_right_full["JT_Ns"] = mb * g * T_contacts_right_full["t_step_s"]

# J1 = (m1 * dv1/dt1 + m1*g) * (2*dt1)
T_contacts_left_full["J1_Ns"] = (
    (m1 * (T_contacts_left_full["dv1_mps"] / T_contacts_left_full["dt1_s"])) + m1 * g
) * (2 * T_contacts_left_full["dt1_s"])

T_contacts_right_full["J1_Ns"] = (
    (m1 * (T_contacts_right_full["dv1_mps"] / T_contacts_right_full["dt1_s"])) + m1 * g
) * (2 * T_contacts_right_full["dt1_s"])

# J2 = JT - J1
T_contacts_left_full["J2_Ns"] = T_contacts_left_full["JT_Ns"] - T_contacts_left_full["J1_Ns"]
T_contacts_right_full["J2_Ns"] = T_contacts_right_full["JT_Ns"] - T_contacts_right_full["J1_Ns"]

T_contacts_left_full.head()

,Step,Touchdown_s,ToeOff_s,NextTouchdown_s,ContactTime_s,AerialTime_s,dt1_s,dv1_mps,t_step_s,JT_Ns,J1_Ns,J2_Ns
0,1,0.405,0.685,1.075,0.280,0.390,0.01,0.330098,0.67,393.04746,4.097001,388.950459
1,2,1.075,1.370,1.745,0.295,0.375,0.01,0.435685,0.67,393.04746,5.107259,387.940201
2,3,1.745,2.030,2.425,0.285,0.395,0.01,0.284125,0.68,398.91384,3.657133,395.256707
3,4,2.425,2.715,3.095,0.290,0.380,0.01,0.228049,0.67,393.04746,3.120595,389.926865
4,5,3.095,3.400,3.775,0.305,0.375,0.01,0.331798,0.68,398.91384,4.113268,394.800572


In [35]:
T_contacts_left_full["F1avg_N"] = T_contacts_left_full["J1_Ns"] / (2 * T_contacts_left_full["dt1_s"])
T_contacts_left_full["F2avg_N"] = T_contacts_left_full["J2_Ns"] / T_contacts_left_full["ContactTime_s"]
T_contacts_left_full["A1_N"] = 2 * T_contacts_left_full["F1avg_N"]

T_contacts_right_full["F1avg_N"] = T_contacts_right_full["J1_Ns"] / (2 * T_contacts_right_full["dt1_s"])
T_contacts_right_full["F2avg_N"] = T_contacts_right_full["J2_Ns"] / T_contacts_right_full["ContactTime_s"]
T_contacts_right_full["A1_N"] = 2 * T_contacts_right_full["F1avg_N"]

print("Left J2 min =", T_contacts_left_full["J2_Ns"].min())
print("Right J2 min =", T_contacts_right_full["J2_Ns"].min())

T_contacts_left_full.head()

Left J2 min = 367.6140765212706
Right J2 min = 384.77619337525533


,Step,Touchdown_s,ToeOff_s,NextTouchdown_s,ContactTime_s,AerialTime_s,dt1_s,dv1_mps,t_step_s,JT_Ns,J1_Ns,J2_Ns,F1avg_N,F2avg_N,A1_N
0,1,0.405,0.685,1.075,0.280,0.390,0.01,0.330098,0.67,393.04746,4.097001,388.950459,204.850061,1389.108781,409.700121
1,2,1.075,1.370,1.745,0.295,0.375,0.01,0.435685,0.67,393.04746,5.107259,387.940201,255.362933,1315.051530,510.725866
2,3,1.745,2.030,2.425,0.285,0.395,0.01,0.284125,0.68,398.91384,3.657133,395.256707,182.856631,1386.865640,365.713261
3,4,2.425,2.715,3.095,0.290,0.380,0.01,0.228049,0.67,393.04746,3.120595,389.926865,156.029769,1344.575395,312.059537
4,5,3.095,3.400,3.775,0.305,0.375,0.01,0.331798,0.68,398.91384,4.113268,394.800572,205.663378,1294.428106,411.326756
